In [28]:
import os
import networkx as nx
from rdkit import Chem
from collections import Counter

In [29]:
print('Loading NCI1 dataset')

graphs = []
y = []

filepath = "datasets/NCI_full/1total-connect.sdf"

supplier = Chem.SDMolSupplier(filepath, removeHs=False)
for mol in supplier:
    if mol is None:
        continue

    G = nx.Graph()

    # Add atoms as nodes
    for atom in mol.GetAtoms():
        G.add_node(
            atom.GetIdx(),
            label=atom.GetSymbol()   # WL uses node labels
        )

    # Add bonds as edges
    for bond in mol.GetBonds():
        G.add_edge(
            bond.GetBeginAtomIdx(),
            bond.GetEndAtomIdx(),
            bond_type=str(bond.GetBondType()),
            bond_order=bond.GetBondTypeAsDouble(),
            aromatic=bond.GetIsAromatic(),
            in_ring=bond.IsInRing(),
            conjugated=bond.GetIsConjugated(),
            stereo=str(bond.GetStereo())
        )

    # Get graph label
    # In NCI1, class label is stored as a molecule property
    label = int(float(mol.GetProp("value")))
    graphs.append(G)
    y.append(label)

print(f"Loaded {len(graphs)} graphs")

Loading NCI1 dataset


[11:43:27] Explicit valence for atom # 1 O, 3, is greater than permitted
[11:43:27] ERROR: Could not sanitize molecule ending on line 1224
[11:43:27] ERROR: Explicit valence for atom # 1 O, 3, is greater than permitted
[11:43:27] Explicit valence for atom # 4 O, 3, is greater than permitted
[11:43:27] ERROR: Could not sanitize molecule ending on line 5612
[11:43:27] ERROR: Explicit valence for atom # 4 O, 3, is greater than permitted
[11:43:27] Explicit valence for atom # 1 Cl, 2, is greater than permitted
[11:43:27] ERROR: Could not sanitize molecule ending on line 5976
[11:43:27] ERROR: Explicit valence for atom # 1 Cl, 2, is greater than permitted
[11:43:27] Explicit valence for atom # 0 Br, 2, is greater than permitted
[11:43:27] ERROR: Could not sanitize molecule ending on line 9488
[11:43:27] ERROR: Explicit valence for atom # 0 Br, 2, is greater than permitted
[11:43:27] Explicit valence for atom # 1 Cl, 2, is greater than permitted
[11:43:27] ERROR: Could not sanitize molecule 

Loaded 36640 graphs


In [27]:
len(graphs)

In [23]:
# ==========================================
# DATASET DETAILS
# ==========================================

num_nodes = [g.number_of_nodes() for g in graphs]
num_edges = [g.number_of_edges() for g in graphs]

print("\n===== DATASET STATISTICS =====")

print(f"Number of graphs : {len(graphs)}")
print(f"Number of nodes: {sum(num_nodes)}")
print(f"Number of edges: {sum(num_edges)}")

print(f"\nNodes per graph")
print(f"Min  : {min(num_nodes)}")
print(f"Max  : {max(num_nodes)}")
print(f"Avg  : {sum(num_nodes)/len(num_nodes):.2f}")

print(f"\nEdges per graph")
print(f"Min  : {min(num_edges)}")
print(f"Max  : {max(num_edges)}")
print(f"Avg  : {sum(num_edges)/len(num_edges):.2f}")

# Label distribution
label_counts = Counter(y)

print("\nLabel Distribution")
for label, count in label_counts.items():
    print(f"Label {label}: {count}")

# ==========================================
# UNIQUE ATOM TYPES
# ==========================================

atom_types = set()

for g in graphs:
    for _, data in g.nodes(data=True):
        atom_types.add(data["label"])

print("\nUnique Atom Types")
print(f"{len(atom_types)}")
print(sorted(atom_types))


===== DATASET STATISTICS =====
Number of graphs : 36640
Number of nodes: 958162
Number of edges: 1038694

Nodes per graph
Min  : 3
Max  : 229
Avg  : 26.15

Edges per graph
Min  : 2
Max  : 236
Avg  : 28.35

Label Distribution
Label 1: 1701
Label -1: 34939

Unique Atom Types
45
['Ac', 'As', 'Au', 'B', 'Bi', 'Br', 'C', 'Cd', 'Cl', 'Co', 'Cr', 'Cu', 'F', 'Fe', 'Ga', 'Ge', 'Hg', 'I', 'Ir', 'K', 'Mg', 'Mn', 'Mo', 'N', 'Na', 'Nd', 'Ni', 'O', 'Os', 'P', 'Pb', 'Pd', 'Pt', 'Re', 'Rh', 'Ru', 'S', 'Sb', 'Se', 'Si', 'Sn', 'Te', 'Ti', 'V', 'Zn']


In [24]:
# Label distribution
label_counts = Counter(y)

print("\nLabel Distribution")
for label, count in label_counts.items():
    print(f"Label {label}: {count}")


# ==========================================
# EDGE TYPES & ATTRIBUTES
# ==========================================

from collections import Counter

edge_type_counter = Counter()
bond_order_counter = Counter()
aromatic_counter = Counter()
ring_counter = Counter()
conjugated_counter = Counter()
stereo_counter = Counter()

for g in graphs:
    for _, _, data in g.edges(data=True):
        edge_type_counter[data["bond_type"]] += 1
        bond_order_counter[data["bond_order"]] += 1
        aromatic_counter[data["aromatic"]] += 1
        ring_counter[data["in_ring"]] += 1
        conjugated_counter[data["conjugated"]] += 1
        stereo_counter[data["stereo"]] += 1

print("\n===== EDGE TYPES & ATTRIBUTES =====")

print("\n--- Edge Types ---")
for edge_type, count in edge_type_counter.items():
    print(f"{edge_type}: {count}")


print("\n--- Bond Orders ---")
for bond_order, count in bond_order_counter.items():
    print(f"{bond_order}: {count}")


print("\n--- Aromatic ---")
for aromatic, count in aromatic_counter.items():
    print(f"{aromatic}: {count}")


print("\n--- Ring ---")
for ring, count in ring_counter.items():
    print(f"{ring}: {count}")


print("\n--- Conjugated ---")
for conjugated, count in conjugated_counter.items():
    print(f"{conjugated}: {count}")


print("\n--- Stereo ---")
for stereo, count in stereo_counter.items():
    print(f"{stereo}: {count}")


Label Distribution
Label 1: 1701
Label -1: 34939

===== EDGE TYPES & ATTRIBUTES =====

--- Edge Types ---
SINGLE: 508892
DOUBLE: 87801
AROMATIC: 438170
TRIPLE: 3831

--- Bond Orders ---
1.0: 508892
2.0: 87801
1.5: 438170
3.0: 3831

--- Aromatic ---
False: 600524
True: 438170

--- Ring ---
False: 402470
True: 636224

--- Conjugated ---
False: 379520
True: 659174

--- Stereo ---
STEREONONE: 1026760
STEREOANY: 11934


In [3]:
y

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,


In [4]:
y_set = set(y)
y_set

{-1, 1}

In [5]:
a = y.count(1)

In [6]:
b = y.count(-1)

In [7]:
a

1773

In [8]:
b

35745

In [9]:
a + b

37518